# Text-to-SQL — QLoRA Fine-tuning + Evaluation

Fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA on the processed dataset, then evaluates zero-shot vs. fine-tuned **execution accuracy** on held-out test examples. Built for Kaggle's Save & Run All (background job, GPU: T4).

In [ ]:
!pip install -q -U transformers peft accelerate trl bitsandbytes datasets

In [ ]:
import os, re, json, random, shutil, sqlite3
from contextlib import nullcontext
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

random.seed(42)

## Load the processed dataset

Copies from the read-only `/kaggle/input` mount to a writable directory first — needed for the tokenizer cache written during training.

In [ ]:
def find_data_dir(root="/kaggle/input"):
    for dirpath, dirnames, _ in os.walk(root):
        if {"train", "val", "test"}.issubset(set(dirnames)):
            return dirpath
    return None

SRC_DATA_DIR = find_data_dir()
assert SRC_DATA_DIR is not None, "No train/val/test found under /kaggle/input"

DATA_DIR = "/kaggle/working/data"
if not os.path.isdir(DATA_DIR):
    shutil.copytree(SRC_DATA_DIR, DATA_DIR)

train_ds = load_from_disk(f"{DATA_DIR}/train")
val_ds = load_from_disk(f"{DATA_DIR}/val")
test_ds = load_from_disk(f"{DATA_DIR}/test")
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## Load base model in 4-bit (QLoRA) + attach LoRA

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = (
    "You are a precise text-to-SQL assistant. Given a database schema and a "
    "natural language question, output ONLY the SQL query that answers it. "
    "No explanation, no markdown, just the query."
)

USE_BF16 = torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
print(f"GPU: {torch.cuda.get_device_name(0)} | precision: {'bf16' if USE_BF16 else 'fp16'}")

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

2 epochs — validation loss plateaus by ~1.5 epochs on this dataset, so a 3rd epoch isn't worth the extra time.

In [ ]:
OUTPUT_DIR = "/kaggle/working/checkpoints"
ADAPTER_DIR = "/kaggle/working/adapter"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=150,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=2,
    report_to="none",
    max_length=512,
    packing=False,
    loss_type="nll",
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
)

last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        last_checkpoint = os.path.join(OUTPUT_DIR, sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1])
        print(f"Resuming from {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
shutil.make_archive("/kaggle/working/text-to-sql-adapter", "zip", ADAPTER_DIR)
print("Adapter saved to /kaggle/working/adapter")

## Evaluate — zero-shot vs. fine-tuned

Execution accuracy on held-out test examples: generate SQL, run it against synthetic sample data matching the schema, and compare to the gold query's result. `model.disable_adapter()` gives the zero-shot pass for free — no second model load needed.

In [ ]:
model.eval()

def parse_messages(messages):
    user = messages[1]["content"]
    gold_sql = messages[2]["content"]
    m = re.match(r"Schema:\n(.*)\n\nQuestion: (.*)", user, re.DOTALL)
    return m.group(1), m.group(2), gold_sql

def clean_sql(text):
    text = re.sub(r"```sql|```", "", text, flags=re.IGNORECASE).strip()
    if ";" in text:
        text = text.split(";")[0]
    elif text.splitlines():
        text = text.splitlines()[0]
    return text.strip()

def generate_sql(context, question, max_new_tokens=128):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Schema:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return clean_sql(tokenizer.decode(gen, skip_special_tokens=True))

In [ ]:
def populate_synthetic_data(conn, create_sql, num_rows=10):
    cur = conn.cursor()
    tables = []
    for stmt in [s.strip() for s in create_sql.split(";") if s.strip()]:
        cur.execute(stmt)
        m = re.search(r"CREATE TABLE\s+(\w+)", stmt, re.IGNORECASE)
        if m:
            tables.append(m.group(1))
    text_pool = ["alpha", "beta", "gamma", "delta", "epsilon"]
    for table in tables:
        cols = cur.execute(f"PRAGMA table_info({table})").fetchall()
        for _ in range(num_rows):
            row = []
            for col in cols:
                t = (col[2] or "").upper()
                if "INT" in t:
                    row.append(random.randint(1, 100))
                elif any(k in t for k in ("REAL", "FLOA", "DOUB", "DEC")):
                    row.append(round(random.uniform(1, 1000), 2))
                else:
                    row.append(random.choice(text_pool) + str(random.randint(1, 20)))
            cur.execute(f"INSERT INTO {table} VALUES ({','.join(['?'] * len(row))})", row)
    conn.commit()

def run_query(conn, sql):
    try:
        return sorted(conn.cursor().execute(sql).fetchall())
    except Exception:
        return None

def execution_match(context, gold_sql, pred_sql, num_rows=10):
    conn = sqlite3.connect(":memory:")
    try:
        populate_synthetic_data(conn, context, num_rows=num_rows)
    except Exception:
        conn.close()
        return None
    gold_result = run_query(conn, gold_sql)
    pred_result = run_query(conn, pred_sql)
    conn.close()
    if gold_result is None:
        return None
    return pred_result is not None and pred_result == gold_result

In [ ]:
NUM_EVAL = 200
eval_examples = test_ds.select(range(min(NUM_EVAL, len(test_ds))))

def evaluate(label, use_adapter):
    records = []
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with ctx:
        for i, ex in enumerate(eval_examples):
            context, question, gold_sql = parse_messages(ex["messages"])
            pred_sql = generate_sql(context, question)
            match = execution_match(context, gold_sql, pred_sql)
            records.append({"question": question, "gold": gold_sql, "pred": pred_sql, "match": match})
            if (i + 1) % 50 == 0:
                print(f"  {label}: {i + 1}/{len(eval_examples)}")
    scored = [r["match"] for r in records if r["match"] is not None]
    accuracy = sum(scored) / len(scored) if scored else 0.0
    print(f"{label}: {accuracy:.1%} ({sum(scored)}/{len(scored)}, {len(records) - len(scored)} excluded)")
    return records, accuracy

zs_records, zs_acc = evaluate("zero-shot", use_adapter=False)
ft_records, ft_acc = evaluate("fine-tuned", use_adapter=True)

print(f"\nZero-shot execution accuracy:  {zs_acc:.1%}")
print(f"Fine-tuned execution accuracy: {ft_acc:.1%}")
print(f"Improvement:                   {(ft_acc - zs_acc) * 100:+.1f} pts")

In [ ]:
results = {
    "num_eval_examples": len(eval_examples),
    "zero_shot_accuracy": zs_acc,
    "fine_tuned_accuracy": ft_acc,
    "improvement_pts": (ft_acc - zs_acc) * 100,
    "zero_shot_examples": zs_records[:10],
    "fine_tuned_examples": ft_records[:10],
}
with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)